# Score-based Generative Modeling (VP-SDE) — Refactored
- Supports **MNIST** and **CIFAR10** via `CFG.dataset`.
- Trains **epsilon-prediction** UNet with time/class conditioning.
- Sampling: **Euler–Maruyama (reverse SDE)** and **Probability Flow ODE**.
- Includes **EMA** for better sample quality.


In [ ]:
# Imports
import math
import os
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [ ]:
# Reproducibility
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)


In [ ]:
# Configuration
@dataclass
class Config:
    # data
    dataset: str = "mnist"      # "mnist" | "cifar10"
    data_root: str = "./data"
    img_size: int = 32
    num_classes: int = 10

    # train
    epochs: int = 20
    batch_size: int = 128
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0

    # sde (VP)
    beta_min: float = 0.1
    beta_max: float = 20.0

    # model
    base_channels: int = 64
    ratios: tuple = (1, 2, 3, 4)
    dropout: float = 0.1

    # sampling
    em_steps: int = 1000     # Euler–Maruyama steps
    ode_steps: int = 200     # Probability flow ODE steps
    n_samples: int = 16
    sample_class: int = 5

    # EMA
    use_ema: bool = True
    ema_decay: float = 0.999

    # system
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    amp: bool = True  # automatic mixed precision (CUDA only)

CFG = Config()
print("device:", CFG.device)


In [ ]:
# Data
def build_dataset_and_loader(cfg: Config):
    if cfg.dataset.lower() == "mnist":
        in_channels = 1
        tfm = transforms.Compose([
            transforms.Resize((cfg.img_size, cfg.img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),  # -> [-1, 1]
        ])
        ds = datasets.MNIST(root=cfg.data_root, train=True, download=True, transform=tfm)

    elif cfg.dataset.lower() == "cifar10":
        in_channels = 3
        tfm = transforms.Compose([
            transforms.Resize((cfg.img_size, cfg.img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # -> [-1, 1]
        ])
        ds = datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=tfm)

    else:
        raise ValueError(f"Unsupported dataset: {cfg.dataset}")

    is_cuda = (cfg.device == "cuda")
    num_workers = min(4, os.cpu_count() or 1)
    loader = DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=is_cuda,
        persistent_workers=(is_cuda and num_workers > 0),
        drop_last=True,
    )
    return ds, loader, in_channels

dataset, train_loader, IN_CHANNELS = build_dataset_and_loader(CFG)
print("dataset:", CFG.dataset, "| in_channels:", IN_CHANNELS, "| batches:", len(train_loader))


## VP-SDE

In [ ]:
class SDE:
    def sde(self, x: torch.Tensor, t: torch.Tensor):
        raise NotImplementedError

    def marginal_prob(self, x0: torch.Tensor, t: torch.Tensor):
        raise NotImplementedError

    def prior_sampling(self, shape, device):
        raise NotImplementedError


class VPSDE(SDE):
    """Variance Preserving SDE:
       dx = -1/2 * beta(t) x dt + sqrt(beta(t)) dW
    """
    def __init__(self, beta_min: float, beta_max: float):
        self.beta_min = beta_min
        self.beta_max = beta_max

    def beta(self, t: torch.Tensor) -> torch.Tensor:
        return self.beta_min + (self.beta_max - self.beta_min) * t

    def beta_integral(self, t: torch.Tensor) -> torch.Tensor:
        # ∫0^t beta(s) ds = beta_min * t + (beta_max - beta_min) * t^2 / 2
        return self.beta_min * t + (self.beta_max - self.beta_min) * (t ** 2) / 2

    def sde(self, x: torch.Tensor, t: torch.Tensor):
        beta_t = self.beta(t).view(-1, 1, 1, 1).to(device=x.device, dtype=x.dtype)
        drift = -0.5 * beta_t * x
        diffusion = beta_t.sqrt()
        return drift, diffusion

    def marginal_prob(self, x0: torch.Tensor, t: torch.Tensor):
        # mean = sqrt(alpha_bar) x0,  std = sqrt(1 - alpha_bar)
        alpha_bar = torch.exp(-self.beta_integral(t)).view(-1, 1, 1, 1).to(device=x0.device, dtype=x0.dtype)
        mean = x0 * alpha_bar.sqrt()
        std = (1.0 - alpha_bar).clamp_min(0.0).sqrt()
        return mean, std

    def prior_sampling(self, shape, device):
        return torch.randn(shape, device=device)


sde = VPSDE(beta_min=CFG.beta_min, beta_max=CFG.beta_max)


## Model (epsilon-prediction UNet)

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, emb_dim: int):
        super().__init__()
        self.emb_dim = emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t in [0, 1], shape: [B]
        half = self.emb_dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device, dtype=t.dtype) / (half - 1)
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)


class AdaGN(nn.Module):
    def __init__(self, in_channels: int, emb_dim: int, num_groups: int = 8):
        super().__init__()
        self.in_channels = in_channels
        self.norm = nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, affine=False)
        self.proj = nn.Linear(emb_dim, in_channels * 2)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        scale, shift = self.proj(emb).chunk(2, dim=1)
        return x * (1.0 + scale[:, :, None, None]) + shift[:, :, None, None]


class AttentionBlock(nn.Module):
    def __init__(self, in_channels: int, num_heads: int = 2, num_groups: int = 8):
        super().__init__()
        self.norm = nn.GroupNorm(num_groups=num_groups, num_channels=in_channels)
        self.attn = nn.MultiheadAttention(embed_dim=in_channels, num_heads=num_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        b, c, hgt, wdt = h.shape
        tokens = h.view(b, c, hgt * wdt).transpose(1, 2)  # [B, HW, C]
        y, _ = self.attn(tokens, tokens, tokens)
        y = y.transpose(1, 2).view(b, c, hgt, wdt)
        return x + y


class ResBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, emb_dim: int, dropout: float = 0.1, num_groups: int = 8):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.nonlinear = nn.SiLU()
        self.norm1 = nn.GroupNorm(num_groups=num_groups, num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = AdaGN(out_channels, emb_dim=emb_dim, num_groups=num_groups)
        self.drop = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.shortcut = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        h = self.nonlinear(h)
        h = self.conv1(h)
        h = self.norm2(h, emb)
        h = self.nonlinear(h)
        h = self.drop(h)
        h = self.conv2(h)
        return self.shortcut(x) + h


class DownStage(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, emb_dim: int, dropout: float, attn: bool = False):
        super().__init__()
        self.attn = attn
        self.rb1 = ResBlock(in_channels, in_channels, emb_dim=emb_dim, dropout=dropout)
        self.rb2 = ResBlock(in_channels, in_channels, emb_dim=emb_dim, dropout=dropout)
        self.attn_blk = AttentionBlock(in_channels) if attn else nn.Identity()
        self.down = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor, emb: torch.Tensor):
        x = self.rb1(x, emb)
        x = self.rb2(x, emb)
        x = self.attn_blk(x)
        skip = x
        x = self.down(x)
        return x, skip


class UpStage(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, emb_dim: int, dropout: float, attn: bool = False):
        super().__init__()
        self.attn = attn
        self.rb1 = ResBlock(in_channels, out_channels, emb_dim=emb_dim, dropout=dropout)
        self.rb2 = ResBlock(out_channels, out_channels, emb_dim=emb_dim, dropout=dropout)
        self.attn_blk = AttentionBlock(out_channels) if attn else nn.Identity()

    def forward(self, x: torch.Tensor, skip: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        x = F.interpolate(x, size=(h * 2, w * 2), mode="nearest")
        x = torch.cat([x, skip], dim=1)
        x = self.rb1(x, emb)
        x = self.rb2(x, emb)
        x = self.attn_blk(x)
        return x


class MidBlock(nn.Module):
    def __init__(self, in_channels: int, emb_dim: int, dropout: float):
        super().__init__()
        self.rb1 = ResBlock(in_channels, in_channels, emb_dim=emb_dim, dropout=dropout)
        self.rb2 = ResBlock(in_channels, in_channels, emb_dim=emb_dim, dropout=dropout)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        x = self.rb1(x, emb)
        x = self.rb2(x, emb)
        return x


class EpsilonUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        base_channels: int = 64,
        ratios=(1, 2, 3, 4),
        num_classes: int = 10,
        dropout: float = 0.1,
    ):
        super().__init__()
        chans = [base_channels * r for r in ratios]
        emb_dim = 4 * base_channels

        self.time_embed = TimeEmbedding(emb_dim=emb_dim)
        self.cond_embed = nn.Embedding(num_classes, embedding_dim=emb_dim)

        self.stem = nn.Conv2d(in_channels, chans[0], kernel_size=3, padding=1)

        self.down = nn.ModuleList([
            DownStage(chans[0], chans[1], emb_dim=emb_dim, dropout=dropout, attn=False),
            DownStage(chans[1], chans[2], emb_dim=emb_dim, dropout=dropout, attn=False),
            DownStage(chans[2], chans[3], emb_dim=emb_dim, dropout=dropout, attn=True),
        ])

        self.mid = MidBlock(chans[3], emb_dim=emb_dim, dropout=dropout)

        self.up = nn.ModuleList([
            UpStage(chans[3] + chans[2], chans[2], emb_dim=emb_dim, dropout=dropout, attn=True),
            UpStage(chans[2] + chans[1], chans[1], emb_dim=emb_dim, dropout=dropout, attn=False),
            UpStage(chans[1] + chans[0], chans[0], emb_dim=emb_dim, dropout=dropout, attn=False),
        ])

        self.head = nn.Sequential(
            nn.GroupNorm(num_groups=8, num_channels=chans[0]),
            nn.SiLU(),
            nn.Conv2d(chans[0], in_channels, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor, cond: Optional[torch.Tensor] = None) -> torch.Tensor:
        temb = self.time_embed(t)
        if cond is None:
            emb = temb
        else:
            emb = temb + self.cond_embed(cond)

        h = self.stem(x)
        skips = []
        for stage in self.down:
            h, s = stage(h, emb)
            skips.append(s)

        h = self.mid(h, emb)

        for stage, s in zip(self.up, reversed(skips)):
            h = stage(h, s, emb)

        return self.head(h)


## Score wrapper + Loss (epsilon prediction)

In [ ]:
class ScoreFunction(nn.Module):
    """score(x,t,c) = ∇x log p_t(x|c) approximated via epsilon model: score = -epsilon / std"""
    def __init__(self, eps_model: nn.Module, sde: SDE, eps: float = 1e-5):
        super().__init__()
        self.eps_model = eps_model
        self.sde = sde
        self.eps = eps

    def forward(self, x: torch.Tensor, t: torch.Tensor, c: Optional[torch.Tensor]):
        eps_hat = self.eps_model(x, t, c)
        _, std = self.sde.marginal_prob(torch.zeros_like(x), t)
        return -eps_hat / (std + self.eps)


class EpsilonLoss(nn.Module):
    def __init__(self, sde: SDE):
        super().__init__()
        self.sde = sde

    def forward(self, x0: torch.Tensor, c: torch.Tensor, eps_model: nn.Module) -> torch.Tensor:
        b = x0.size(0)
        t = torch.rand(b, device=x0.device, dtype=x0.dtype).clamp(1e-5, 1.0)
        noise = torch.randn_like(x0)
        mean, std = self.sde.marginal_prob(x0, t)
        xt = mean + std * noise
        noise_hat = eps_model(xt, t, c)
        return (noise - noise_hat).pow(2).mean()


## EMA (optional but recommended)

In [ ]:
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for k, v in msd.items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


## Samplers

In [ ]:
class EulerMaruyamaSampler:
    """Reverse-time SDE sampler (Euler–Maruyama)."""
    def __init__(self, steps: int = 1000):
        self.steps = steps

    @torch.inference_mode()
    def sample(self, score_fn: nn.Module, sde: SDE, shape, cond: Optional[torch.Tensor], device):
        x = sde.prior_sampling(shape, device=device)
        t_grid = torch.linspace(1.0, 1e-5, self.steps + 1, device=device)
        dt = 1.0 / self.steps

        for i in range(self.steps):
            t = t_grid[i].expand(shape[0])
            drift, diff = sde.sde(x, t)
            score = score_fn(x, t, cond)
            rev_drift = drift - (diff ** 2) * score
            x = x - rev_drift * dt + diff * torch.randn_like(x) * math.sqrt(dt)

        return x


class ProbabilityFlowODESampler:
    """Probability flow ODE sampler (deterministic)."""
    def __init__(self, steps: int = 200):
        self.steps = steps

    @torch.inference_mode()
    def sample(self, score_fn: nn.Module, sde: SDE, shape, cond: Optional[torch.Tensor], device):
        x = sde.prior_sampling(shape, device=device)
        t_grid = torch.linspace(1.0, 1e-5, self.steps + 1, device=device)
        dt = 1.0 / self.steps

        for i in range(self.steps):
            t = t_grid[i].expand(shape[0])
            drift, diff = sde.sde(x, t)
            score = score_fn(x, t, cond)
            ode_drift = drift - 0.5 * (diff ** 2) * score
            x = x - ode_drift * dt

        return x


em_sampler = EulerMaruyamaSampler(steps=CFG.em_steps)
ode_sampler = ProbabilityFlowODESampler(steps=CFG.ode_steps)


## Training

In [ ]:
eps_model = EpsilonUNet(
    in_channels=IN_CHANNELS,
    base_channels=CFG.base_channels,
    ratios=CFG.ratios,
    num_classes=CFG.num_classes,
    dropout=CFG.dropout,
).to(CFG.device)

score_fn = ScoreFunction(eps_model, sde).to(CFG.device)
loss_fn = EpsilonLoss(sde).to(CFG.device)

optimizer = torch.optim.AdamW(eps_model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(CFG.amp and CFG.device == "cuda"))

ema = EMA(eps_model, decay=CFG.ema_decay) if CFG.use_ema else None


In [ ]:
def train_one_epoch(cfg: Config, model: nn.Module, loader: DataLoader, optimizer, loss_fn: nn.Module, scaler, ema: Optional[EMA]):
    model.train()
    running = 0.0

    for step, (x0, c) in enumerate(loader, start=1):
        x0 = x0.to(cfg.device, non_blocking=True)
        c = c.to(cfg.device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.amp and cfg.device == "cuda")):
            loss = loss_fn(x0, c, model)

        scaler.scale(loss).backward()

        if cfg.grad_clip is not None and cfg.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        if ema is not None:
            ema.update(model)

        running += float(loss.item())
        if step % 100 == 0:
            print(f"step {step:>5}/{len(loader)} | loss {running/100:.6f}")
            running = 0.0


In [ ]:
for epoch in range(1, CFG.epochs + 1):
    print(f"===== EPOCH {epoch}/{CFG.epochs} =====")
    train_one_epoch(CFG, eps_model, train_loader, optimizer, loss_fn, scaler, ema)


## Sampling + Visualization

In [ ]:
def denorm_to_01(x: torch.Tensor) -> torch.Tensor:
    # x assumed in [-1, 1]
    return (x.clamp(-1, 1) + 1) * 0.5


@torch.inference_mode()
def sample_images(cfg: Config, sampler, use_ema: bool = True):
    # build cond
    cond = torch.full((cfg.n_samples,), int(cfg.sample_class), dtype=torch.long, device=cfg.device)
    shape = (cfg.n_samples, IN_CHANNELS, cfg.img_size, cfg.img_size)

    # choose model (EMA or current)
    if use_ema and ema is not None:
        tmp = EpsilonUNet(
            in_channels=IN_CHANNELS,
            base_channels=cfg.base_channels,
            ratios=cfg.ratios,
            num_classes=cfg.num_classes,
            dropout=cfg.dropout,
        ).to(cfg.device)
        ema.copy_to(tmp)
        local_score = ScoreFunction(tmp, sde).to(cfg.device)
        tmp.eval()
    else:
        eps_model.eval()
        local_score = score_fn

    x = sampler.sample(local_score, sde, shape=shape, cond=cond, device=cfg.device)
    x01 = denorm_to_01(x).detach().cpu()
    return x01


def show_grid(x01: torch.Tensor, nrow: int = 4):
    # x01: [B,C,H,W] in [0,1]
    b, c, h, w = x01.shape
    ncol = int(math.ceil(b / nrow))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.2, nrow * 2.2))
    axes = axes.reshape(nrow, ncol)

    idx = 0
    for r in range(nrow):
        for col in range(ncol):
            ax = axes[r, col]
            ax.axis("off")
            if idx < b:
                img = x01[idx]
                if c == 1:
                    ax.imshow(img[0], cmap="gray")
                else:
                    ax.imshow(img.permute(1, 2, 0))
            idx += 1
    plt.tight_layout()
    plt.show()


In [ ]:
# Probability Flow ODE sampling (deterministic)
samples_ode = sample_images(CFG, ode_sampler, use_ema=True)
show_grid(samples_ode, nrow=4)


In [ ]:
# Euler–Maruyama sampling (stochastic) — usually slower but can yield different characteristics
# samples_em = sample_images(CFG, em_sampler, use_ema=True)
# show_grid(samples_em, nrow=4)


## Debug utilities

In [ ]:
def print_params_by_layer(model: nn.Module) -> None:
    total = 0
    for name, p in model.named_parameters():
        n = p.numel()
        total += n
        print(f"{name:60s} | {list(p.shape)!s:20s} | {n:>10d}")
    print("-" * 100)
    print(f"Total params: {total:,}")

# print_params_by_layer(eps_model)
